# TOBE SML Dataset Pipeline Demo

데이터셋 파트만 셀 단위로 확인하기 위한 데모 노트북입니다.

흐름:
1. 태스크 선택
2. 데이터베이스에서 데이터 로드
3. `head()` 및 메타데이터 확인
4. 태스크별 타겟 검증
5. EDA 수행
6. 전처리 수행
7. 유의성 검증
8. pkl 버전 저장

모델링 파트는 아직 포함하지 않습니다.

## 0. 프로젝트 경로 설정

노트북은 `notebooks/` 아래에 있기 때문에 프로젝트 루트를 Python path에 추가합니다.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
SML_DATASET_PACKAGE = PROJECT_ROOT / "packages" / "sml-dataset"
if str(SML_DATASET_PACKAGE) not in sys.path:
    sys.path.insert(0, str(SML_DATASET_PACKAGE))

PROJECT_ROOT

WindowsPath('c:/work/project/workspace/test2')

## 1. 데모 데이터베이스 생성

실제 운영에서는 DB 접속 정보와 쿼리로 대체될 부분입니다. 데모에서는 SQLite DB를 생성합니다.

In [ ]:
import runpy

runpy.run_path(PROJECT_ROOT / "scripts" / "create_demo_database.py", run_name="__main__")

Created demo database: data\demo_sml.db


{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': WindowsPath('c:/work/project/workspace/test2/scripts/create_demo_database.py'),
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__(name, globals=Non

## 2. 태스크 및 데이터 소스 선택

아래 `CONFIG_PATH`만 바꾸면 이진분류, 단일 회귀, 멀티 회귀를 같은 흐름으로 확인할 수 있습니다.

- `../configs/binary_classification.yaml`
- `../configs/single_regression.yaml`
- `../configs/multi_regression.yaml`

In [ ]:
from sml_dataset.config import PipelineConfig

CONFIG_PATH = PROJECT_ROOT / "configs" / "binary_classification.yaml"
config = PipelineConfig.from_yaml(CONFIG_PATH)

config

PipelineConfig(task=<TaskType.BINARY_CLASSIFICATION: 'binary_classification'>, data_source=DataSourceConfig(type='sqlite', options={'database': 'data/demo_sml.db', 'query': 'SELECT * FROM customer_churn'}), targets=['churn'], preprocessing={'drop_duplicates': True, 'missing': {'numeric': 'median', 'categorical': 'most_frequent'}, 'outliers': {'method': 'iqr_clip', 'factor': 1.5}, 'encoding': {'method': 'onehot'}, 'scaling': {'method': 'standard'}}, versioning=VersioningConfig(artifact_dir=WindowsPath('artifacts/datasets'), name='customer_churn_binary'))

## 3. 데이터베이스에서 데이터 로드

In [ ]:
from sml_dataset.connectors import create_connector

connector = create_connector(config.data_source.type, config.data_source.options)
raw_df = connector.load()

raw_df.shape

(120, 7)

## 4. 데이터 미리보기

In [ ]:
raw_df.head()

,customer_id,age,monthly_fee,tenure_months,contract_type,region,churn
0,1,23,49.42,16,annual,daegu,0
1,2,62,82.43,64,monthly,seoul,1
2,3,55,34.71,9,monthly,daegu,1
3,4,43,58.97,37,annual,seoul,0
4,5,42,67.93,34,two_year,incheon,0


## 5. 사용자에게 보여줄 메타데이터 생성

In [ ]:
import pandas as pd
from sml_dataset.metadata import build_metadata

metadata = build_metadata(raw_df)
pd.DataFrame(metadata["columns"])

,name,dtype,missing_count,missing_ratio,unique_count
0,customer_id,int64,0,0.000000,120
1,age,int64,0,0.000000,49
2,monthly_fee,float64,1,0.008333,118
3,tenure_months,int64,0,0.000000,60
4,contract_type,str,0,0.000000,3
5,region,str,1,0.008333,4
6,churn,int64,0,0.000000,2


In [ ]:
pd.DataFrame(metadata["preview"])

,customer_id,age,monthly_fee,tenure_months,contract_type,region,churn
0,1,23,49.42,16,annual,daegu,0
1,2,62,82.43,64,monthly,seoul,1
2,3,55,34.71,9,monthly,daegu,1
3,4,43,58.97,37,annual,seoul,0
4,5,42,67.93,34,two_year,incheon,0


## 6. 태스크별 타겟 검증

이진분류는 타겟 고유값이 정확히 2개여야 합니다.
회귀는 타겟이 숫자형이어야 하며, 단일 회귀는 1개, 멀티 회귀는 2개 이상이어야 합니다.

In [ ]:
from sml_dataset.target_validation import validate_targets

validate_targets(raw_df, config.task, config.targets)
print(f"Target validation passed: task={config.task.value}, targets={config.targets}")

Target validation passed: task=binary_classification, targets=['churn']


In [ ]:
raw_df[config.targets].head()

,churn
0,0
1,1
2,1
3,0
4,0


## 7. EDA 수행

EDA 결과는 수치 요약과 UI 렌더링용 차트 스펙을 함께 반환합니다.

In [ ]:
from sml_dataset.eda import run_eda

eda = run_eda(raw_df, config.task, config.targets)
eda.keys()

dict_keys(['task', 'rows', 'columns', 'numeric_columns', 'categorical_columns', 'chart_specs', 'missing_by_column', 'duplicate_rows', 'numeric_summary', 'categorical_summary', 'target_distribution'])

In [ ]:
pd.Series(eda["missing_by_column"], name="missing_count").to_frame()

,missing_count
customer_id,0
age,0
monthly_fee,1
tenure_months,0
contract_type,0
region,1
churn,0


In [ ]:
pd.DataFrame(eda["chart_specs"]).head(20)

,type,title,x,y
0,histogram,customer_id distribution,customer_id,NaN
1,histogram,age distribution,age,NaN
2,histogram,monthly_fee distribution,monthly_fee,NaN
3,histogram,tenure_months distribution,tenure_months,NaN
4,histogram,churn distribution,churn,NaN
5,bar,contract_type frequency,contract_type,count
6,bar,region frequency,region,count
7,bar,churn distribution,churn,count
8,box,customer_id by churn,churn,customer_id
9,box,age by churn,churn,age


## 8. EDA 차트 예시

운영 화면에서는 `chart_specs`를 Plotly, ECharts, Chart.js 등으로 렌더링하면 됩니다. 노트북에서는 pandas plot으로 간단히 확인합니다.

In [ ]:
numeric_cols = raw_df.select_dtypes(include="number").columns.tolist()
if numeric_cols:
    raw_df[numeric_cols[0]].hist(bins=20)

ImportError: matplotlib is required for plotting when the default backend "matplotlib" is selected.

In [ ]:
if config.task.value == "binary_classification":
    raw_df[config.targets[0]].value_counts().plot(kind="bar")

## 9. 전처리 수행

포함된 전처리:
- 중복 데이터 제거
- 결측치 처리
- 이상치 IQR clipping
- 원핫 인코딩 또는 레이블 인코딩
- Robust, Standard, MinMax, None 스케일링

In [ ]:
from sml_dataset.preprocessing import preprocess_dataset

preprocess_result = preprocess_dataset(raw_df, config.targets, config.preprocessing)

preprocess_result.report

In [ ]:
preprocess_result.features.head()

In [ ]:
preprocess_result.targets.head()

## 10. 태스크별 유의성 검증

- 이진분류: 숫자형 feature는 Welch t-test, 범주형 feature는 chi-square test
- 회귀: 숫자형 feature와 target 간 Pearson correlation

In [ ]:
from sml_dataset.significance import run_significance_tests

significance = run_significance_tests(raw_df, config.task, config.targets)
significance.keys()

In [ ]:
if "numeric_tests" in significance:
    display(pd.DataFrame(significance["numeric_tests"]).T)
if "categorical_tests" in significance:
    display(pd.DataFrame(significance["categorical_tests"]).T)
if "pearson_correlations" in significance:
    for target, values in significance["pearson_correlations"].items():
        print(target)
        display(pd.DataFrame(values).T)

## 11. pkl 버전 저장

전처리된 feature, target, metadata, preprocessing report, preprocessor 객체를 하나의 버전 파일로 저장합니다.

In [ ]:
from dataclasses import asdict
from sml_dataset.versioning import save_dataset_version

version_info = save_dataset_version(
    name=config.versioning.name,
    artifact_dir=config.versioning.artifact_dir,
    features=preprocess_result.features,
    targets=preprocess_result.targets,
    metadata=metadata,
    preprocessing_report=preprocess_result.report,
    preprocessor=preprocess_result.preprocessor,
)

asdict(version_info)

## 12. 전체 파이프라인으로 한 번에 실행

위 셀 단위 흐름이 안정화되면 아래처럼 py 모듈의 `DatasetPipeline`로 묶어서 실행할 수 있습니다.

In [ ]:
from sml_dataset.pipeline import DatasetPipeline

bundle = DatasetPipeline(config).run()

{
    "raw_shape": bundle.raw.shape,
    "feature_shape": bundle.features.shape,
    "target_shape": bundle.targets.shape,
    "version": asdict(bundle.version_info),
}